# Seed-Always-Wins Brier Score Baseline

Compare prediction strategies on **all available actual tournament games**:

| Strategy | Evaluated on |
|---|---|
| **Seed-always-wins** | All historical games (Men's 1985–2025, Women's 1998–2025) |
| **LR model** | Submission seasons only (2023–2025) — the only seasons it covers |

**Brier score** = `mean((pred − actual)²)` — lower is better (0.0 = perfect, 0.25 = always 0.5)

> Note: The submission template contains ~12K *hypothetical* matchup pairs. Only the ~63 games
> per bracket that actually happened have real outcomes — those are what we evaluate here.

In [12]:
import re
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR   = Path("../data/landing")
MENS_DIR   = DATA_DIR / "mens"
WOMENS_DIR = DATA_DIR / "womens"

# Verify files present
for p in [
    MENS_DIR   / "MNCAATourneySeeds.csv",
    MENS_DIR   / "MNCAATourneyCompactResults.csv",
    WOMENS_DIR / "WNCAATourneySeeds.csv",
    WOMENS_DIR / "WNCAATourneyCompactResults.csv",
    DATA_DIR   / "SampleSubmissionStage1.csv",
    DATA_DIR.parent / "submission_simple.csv",
]:
    assert p.exists(), f"Missing: {p}"
    print(f"✓ {p.name}")


✓ MNCAATourneySeeds.csv
✓ MNCAATourneyCompactResults.csv
✓ WNCAATourneySeeds.csv
✓ WNCAATourneyCompactResults.csv
✓ SampleSubmissionStage1.csv
✓ submission_simple.csv


In [13]:
# ── Load seeds and results ────────────────────────────────────────────────────

def load_gender(seeds_path, results_path, gender):
    seeds   = pl.read_csv(seeds_path).with_columns(pl.lit(gender).alias("Gender"))
    results = pl.read_csv(results_path).with_columns(pl.lit(gender).alias("Gender"))
    return seeds, results

m_seeds, m_results = load_gender(MENS_DIR/"MNCAATourneySeeds.csv",
                                  MENS_DIR/"MNCAATourneyCompactResults.csv", "M")
w_seeds, w_results = load_gender(WOMENS_DIR/"WNCAATourneySeeds.csv",
                                  WOMENS_DIR/"WNCAATourneyCompactResults.csv", "W")

seeds   = pl.concat([m_seeds,   w_seeds])
results = pl.concat([m_results, w_results])

# Parse seed strings → integer (e.g. 'W01' → 1, 'Z16a' → 16)
seeds = seeds.with_columns(
    pl.col("Seed").str.extract(r"(\d+)", 1).cast(pl.Int32).alias("SeedNum")
)

print(f"Seeds:   {seeds.shape}   Seasons: {seeds['Season'].min()}–{seeds['Season'].max()}")
print(f"Results: {results.shape}  Seasons: {results['Season'].min()}–{results['Season'].max()}")


Seeds:   (4416, 5)   Seasons: 1985–2025
Results: (4347, 9)  Seasons: 1985–2025


In [14]:
# ── Determine submission seasons (used later for LR model comparison) ─────────

sub_template = pl.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub_seasons  = sorted(
    sub_template.with_columns(
        pl.col("ID").str.split("_").list.get(0).cast(pl.Int32).alias("Season")
    )["Season"].unique().to_list()
)
print(f"Submission seasons (for LR comparison): {sub_seasons}")

# ── Use ALL available seasons for the seed-always-wins baseline ───────────────
# (no season filter — use every game in the data)

print(f"\nAll results: {len(results)} games across seasons {results['Season'].min()}–{results['Season'].max()}")
print(results.group_by("Gender").agg(
    pl.len().alias("n_games"),
    pl.col("Season").min().alias("first_season"),
    pl.col("Season").max().alias("last_season"),
).sort("Gender"))


Submission seasons (for LR comparison): [2023, 2024, 2025]

All results: 4347 games across seasons 1985–2025
shape: (2, 4)
┌────────┬─────────┬──────────────┬─────────────┐
│ Gender ┆ n_games ┆ first_season ┆ last_season │
│ ---    ┆ ---     ┆ ---          ┆ ---         │
│ str    ┆ u32     ┆ i64          ┆ i64         │
╞════════╪═════════╪══════════════╪═════════════╡
│ M      ┆ 2583    ┆ 1985         ┆ 2025        │
│ W      ┆ 1764    ┆ 1998         ┆ 2025        │
└────────┴─────────┴──────────────┴─────────────┘


In [15]:
# ── Build actual outcome rows with submission-style IDs ───────────────────────
#
# ID convention: {Season}_{Team1}_{Team2}  where Team1 = lower TeamID
# Actual outcome: 1 if Team1 (lower ID) won, 0 if Team2 (higher ID) won
# We use ALL games (no season filter)

actuals_labeled = results.with_columns([
    pl.when(pl.col("WTeamID") < pl.col("LTeamID"))
      .then(pl.col("WTeamID")).otherwise(pl.col("LTeamID")).alias("T1"),
    pl.when(pl.col("WTeamID") < pl.col("LTeamID"))
      .then(pl.col("LTeamID")).otherwise(pl.col("WTeamID")).alias("T2"),
    pl.when(pl.col("WTeamID") < pl.col("LTeamID"))
      .then(pl.lit(1)).otherwise(pl.lit(0)).cast(pl.Float64).alias("Actual"),
]).with_columns(
    (pl.col("Season").cast(pl.Utf8) + "_"
     + pl.col("T1").cast(pl.Utf8) + "_"
     + pl.col("T2").cast(pl.Utf8)).alias("ID")
).select(["ID", "Season", "Gender", "T1", "T2", "Actual"])

print(f"Total labeled actual games: {len(actuals_labeled)}")
print(actuals_labeled.group_by("Gender").agg(
    pl.len().alias("n_games"),
    pl.col("Season").min().alias("first"),
    pl.col("Season").max().alias("last"),
).sort("Gender"))
actuals_labeled.head(3)


Total labeled actual games: 4347
shape: (2, 4)
┌────────┬─────────┬───────┬──────┐
│ Gender ┆ n_games ┆ first ┆ last │
│ ---    ┆ ---     ┆ ---   ┆ ---  │
│ str    ┆ u32     ┆ i64   ┆ i64  │
╞════════╪═════════╪═══════╪══════╡
│ M      ┆ 2583    ┆ 1985  ┆ 2025 │
│ W      ┆ 1764    ┆ 1998  ┆ 2025 │
└────────┴─────────┴───────┴──────┘


ID,Season,Gender,T1,T2,Actual
str,i64,str,i64,i64,f64
"""1985_1116_1160""",1985,"""M""",1116,1160,0.0
"""1985_1126_1152""",1985,"""M""",1126,1152,1.0
"""1985_1124_1128""",1985,"""M""",1124,1128,1.0


In [16]:
# ── Join seed numbers for T1 and T2 ──────────────────────────────────────────

seed_lookup = seeds.select(["Season", "Gender", "TeamID", "SeedNum"])

eval_df = (
    actuals_labeled
    .join(
        seed_lookup.rename({"TeamID": "T1", "SeedNum": "T1_Seed"}),
        on=["Season", "Gender", "T1"],
        how="left",
    )
    .join(
        seed_lookup.rename({"TeamID": "T2", "SeedNum": "T2_Seed"}),
        on=["Season", "Gender", "T2"],
        how="left",
    )
)

missing_seeds = eval_df.filter(pl.col("T1_Seed").is_null() | pl.col("T2_Seed").is_null()).height
print(f"Games with missing seeds (will use 0.5): {missing_seeds}")
print(f"Games with full seed info: {len(eval_df) - missing_seeds}")

# ── Seed-always-wins prediction ───────────────────────────────────────────────
#
# Lower seed number = better team. e.g. seed 1 beats seed 16.
# If T1_Seed < T2_Seed  → T1 is better → predict 1.0
# If T1_Seed > T2_Seed  → T2 is better → predict 0.0
# If tied or missing    → predict 0.5

eval_df = eval_df.with_columns(
    pl.when(pl.col("T1_Seed") < pl.col("T2_Seed")).then(pl.lit(1.0))
      .when(pl.col("T1_Seed") > pl.col("T2_Seed")).then(pl.lit(0.0))
      .otherwise(pl.lit(0.5))
      .alias("Pred_SeedWins")
)

print("\nSeed-wins prediction distribution:")
print(eval_df["Pred_SeedWins"].value_counts().sort("Pred_SeedWins"))


Games with missing seeds (will use 0.5): 0
Games with full seed info: 4347

Seed-wins prediction distribution:
shape: (3, 2)
┌───────────────┬───────┐
│ Pred_SeedWins ┆ count │
│ ---           ┆ ---   │
│ f64           ┆ u32   │
╞═══════════════╪═══════╡
│ 0.0           ┆ 1719  │
│ 0.5           ┆ 937   │
│ 1.0           ┆ 1691  │
└───────────────┴───────┘


In [17]:
# ── Load our LR model predictions and join to actual games ────────────────────

lr_preds = pl.read_csv(DATA_DIR.parent / "submission_simple.csv")  # ../data/submission_simple.csv

eval_df = eval_df.join(
    lr_preds.rename({"Pred": "Pred_LR"}),
    on="ID",
    how="left",
)

missing_lr = eval_df.filter(pl.col("Pred_LR").is_null()).height
print(f"Actual games NOT covered by LR submission: {missing_lr}")
print(f"  (these will be excluded from LR Brier score)")

# Keep only games where both predictions are available
eval_full = eval_df.drop_nulls(subset=["Pred_LR"])
print(f"\nGames used for comparison: {len(eval_full)}")
eval_full.head(5)


Actual games NOT covered by LR submission: 3969
  (these will be excluded from LR Brier score)

Games used for comparison: 378


ID,Season,Gender,T1,T2,Actual,T1_Seed,T2_Seed,Pred_SeedWins,Pred_LR
str,i64,str,i64,i64,f64,i32,i32,f64,f64
"""2023_1101_1160""",2023,"""M""",1101,1160,1.0,1,4,1.0,0.162328
"""2023_1108_1150""",2023,"""M""",1108,1150,1.0,1,4,1.0,0.895078
"""2023_1111_1136""",2023,"""M""",1111,1136,1.0,1,4,1.0,0.468261
"""2023_1120_1122""",2023,"""M""",1120,1122,1.0,4,1,0.0,0.707263
"""2023_1107_1133""",2023,"""M""",1107,1133,1.0,2,3,1.0,0.10474


In [18]:
# ── Compute Brier Scores ──────────────────────────────────────────────────────

def brier_score(preds: pl.Series, actuals: pl.Series) -> float:
    return float(((preds - actuals) ** 2).mean())

# ── Section 1: Seed-always-wins on ALL historical games ───────────────────────

rows = []
for split, df in [
    ("Men's (1985–2025)",   eval_df.filter(pl.col("Gender") == "M")),
    ("Women's (1998–2025)", eval_df.filter(pl.col("Gender") == "W")),
    ("All genders",         eval_df),
]:
    rows.append({
        "Split":          split,
        "n_games":        len(df),
        "SeedWins_BS":    round(brier_score(df["Pred_SeedWins"], df["Actual"]), 4),
        "Always_0.5_BS":  0.25,
    })

print("=" * 62)
print("SECTION 1 — Seed-Always-Wins on ALL historical games")
print("=" * 62)
print(pd.DataFrame(rows).to_string(index=False))

# ── Section 2: LR model vs seed-wins on submission seasons only ───────────────

eval_sub = eval_df.filter(pl.col("Season").is_in(sub_seasons))
eval_lr  = eval_sub.drop_nulls(subset=["Pred_LR"])  # 2 games missing coverage

rows2 = []
for split, df in [
    ("Men's (2023–2025)",   eval_lr.filter(pl.col("Gender") == "M")),
    ("Women's (2023–2025)", eval_lr.filter(pl.col("Gender") == "W")),
    ("Both genders",        eval_lr),
]:
    bs_sw = round(brier_score(df["Pred_SeedWins"], df["Actual"]), 4)
    bs_lr = round(brier_score(df["Pred_LR"],       df["Actual"]), 4)
    rows2.append({
        "Split":          split,
        "n_games":        len(df),
        "SeedWins_BS":    bs_sw,
        "LR_model_BS":    bs_lr,
        "Delta":          f"{'▼' if bs_lr < bs_sw else '▲'}{abs(bs_lr - bs_sw):.4f}",
    })

print("\n" + "=" * 62)
print("SECTION 2 — LR model vs Seed-Always-Wins (2023–2025 only)")
print("=" * 62)
print(pd.DataFrame(rows2).to_string(index=False))
print("\n✓ LR beats seed-wins" if brier_score(eval_lr["Pred_LR"], eval_lr["Actual"])
      < brier_score(eval_lr["Pred_SeedWins"], eval_lr["Actual"]) else
      "✗ Seed-wins beats LR")


SECTION 1 — Seed-Always-Wins on ALL historical games
              Split  n_games  SeedWins_BS  Always_0.5_BS
  Men's (1985–2025)     2583       0.3617           0.25
Women's (1998–2025)     1764       0.3662           0.25
        All genders     4347       0.3635           0.25

SECTION 2 — LR model vs Seed-Always-Wins (2023–2025 only)
              Split  n_games  SeedWins_BS  LR_model_BS   Delta
  Men's (2023–2025)      189       0.3492       0.3704 ▲0.0212
Women's (2023–2025)      189       0.3492       0.3209 ▼0.0283
       Both genders      378       0.3492       0.3457 ▼0.0035

✓ LR beats seed-wins


## Summary

### Baselines (over all historical games)

| Strategy | n_games | Brier Score |
|---|---|---|
| **Seed-always-wins (Men's, 1985–2025)** | 2,583 | 0.3617 |
| **Seed-always-wins (Women's, 1998–2025)** | 1,764 | 0.3662 |
| **Seed-always-wins (All, 1985–2025)** | 4,347 | **0.3635** |
| **Always predict 0.5** | — | 0.2500 |

### LR model vs baselines (2023–2025 only)

| Split | Seed-wins BS | LR model BS | Δ |
|---|---|---|---|
| Men's | 0.3492 | 0.3704 | ▲ 0.0212 worse |
| Women's | 0.3492 | 0.3209 | ▼ 0.0283 better |
| **Both** | **0.3492** | **0.3457** | ▼ 0.0035 better |

### Key takeaways
- **Seed-always-wins ≈ 0.36** across all eras — the hard ceiling any model must beat
- The LR model barely squeaks past seed-wins overall (+0.004) — **men's is actually worse** than just picking the better seed
- The "always 0.5" floor (0.25) shows how much room for improvement remains
- Target for a good model: Brier score well below 0.25 (probabilistic soft predictions help a lot)

> With real data going back to 1985, these baselines will be much more statistically robust (~63 games/season × 41 seasons for men's).